#**Estadística descriptiva e histogramas**

La estadística descriptiva es el primer paso en cualquier análisis de datos, ya que permite resumir, organizar y entender la información de manera general antes de aplicar métodos más complejos. Su objetivo principal es describir las características más importantes de los datos mediante medidas como la media, la mediana, la desviación estándar, los valores mínimos y máximos, entre otros.
Sin embargo, las medidas numéricas por sí solas no siempre son suficientes para comprender completamente los datos. Por ello, es fundamental complementarlas con herramientas visuales, como los histogramas, que permiten observar la distribución de una variable de forma gráfica.
El histograma es especialmente útil cuando se trabaja con variables numéricas, ya que agrupa los valores en intervalos y muestra cuántas observaciones caen en cada uno. Esto facilita la identificación de patrones, como la concentración de valores en ciertos rangos, la presencia de asimetría o la existencia de valores atípicos.
En el contexto clínico, combinar estadística descriptiva con histogramas permite tener una visión más completa de los datos. Por ejemplo, mientras que un promedio puede indicar el nivel general de glucosa en una población, el histograma puede revelar si existen muchos pacientes con valores elevados o si la distribución es desigual.

In [ ]:

install.packages("tidyverse")
install.packages("janitor")
install.packages("gtsummary")


library(tidyverse)
library(janitor)
library(gtsummary)

datos <- read_csv("data/datos_medicos.csv") %>%
  clean_names() %>%
  mutate(
    sexo = factor(sexo),
    grupo = factor(grupo),
    fumador = factor(fumador),
    diabetes = factor(diabetes, levels = c(0, 1), labels = c("No", "Sí")),
    hipertension = factor(hipertension, levels = c(0, 1), labels = c("No", "Sí")),
    evento_complicacion = factor(evento_complicacion, levels = c(0, 1), labels = c("No", "Sí")),
    cambio_hb = hb_post - hb_pre
  )

In [ ]:
# ----------------------------------------------------------
# 1. Descriptivos básicos
# ----------------------------------------------------------

mean(datos$edad, na.rm = TRUE)
sd(datos$edad, na.rm = TRUE)
median(datos$edad, na.rm = TRUE)
quantile(datos$edad, probs = c(0.25, 0.75), na.rm = TRUE)
range(datos$edad, na.rm = TRUE)

In [ ]:
# ----------------------------------------------------------
# 2. Función útil para resumir variable numérica
# ----------------------------------------------------------

resumen_numerico <- function(x) {
  tibble(
    n = sum(!is.na(x)),
    media = mean(x, na.rm = TRUE),
    de = sd(x, na.rm = TRUE),
    mediana = median(x, na.rm = TRUE),
    q1 = quantile(x, 0.25, na.rm = TRUE),
    q3 = quantile(x, 0.75, na.rm = TRUE),
    minimo = min(x, na.rm = TRUE),
    maximo = max(x, na.rm = TRUE)
  )
}

resumen_numerico(datos$glucosa_mg_dl)

In [ ]:
# ----------------------------------------------------------
# 3. Descriptivos por grupo
# ----------------------------------------------------------

datos %>%
  group_by(grupo) %>%
  summarise(
    n = n(),
    edad_media = mean(edad, na.rm = TRUE),
    edad_de = sd(edad, na.rm = TRUE),
    glucosa_mediana = median(glucosa_mg_dl, na.rm = TRUE),
    glucosa_q1 = quantile(glucosa_mg_dl, 0.25, na.rm = TRUE),
    glucosa_q3 = quantile(glucosa_mg_dl, 0.75, na.rm = TRUE),
    complicaciones = sum(evento_complicacion == "Sí", na.rm = TRUE),
    prop_complicaciones = mean(evento_complicacion == "Sí", na.rm = TRUE)
  )

In [ ]:
# ----------------------------------------------------------
# 4. Variables categóricas: frecuencias y porcentajes
# ----------------------------------------------------------

tabyl(datos, diabetes)
tabyl(datos, grupo, diabetes) %>%
  adorn_percentages("row") %>%
  adorn_pct_formatting(digits = 1)

In [ ]:
# ----------------------------------------------------------
# 5. Tabla clínica tipo "Tabla 1"
# ----------------------------------------------------------

tabla_1 <- datos %>%
  select(edad, sexo, grupo, imc, diabetes, hipertension, glucosa_mg_dl, evento_complicacion) %>%
  tbl_summary(
    by = grupo,
    statistic = list(
      all_continuous() ~ "{mean} ± {sd}",
      all_categorical() ~ "{n} ({p}%)"
    ),
    missing = "ifany"
  ) %>%
  add_overall() %>%
  add_p()

print(tabla_1)

# Asegúrate de tener cargada esta librería
library(htmltools)

tabla_1 <- datos %>%
  select(edad, sexo, grupo, imc, diabetes, hipertension, glucosa_mg_dl, evento_complicacion) %>%
  tbl_summary(
    by = grupo,
    statistic = list(
      all_continuous() ~ "{mean} ± {sd}",
      all_categorical() ~ "{n} ({p}%)"
    ),
    missing = "ifany"
  ) %>%
  add_overall() %>%
  add_p() %>%
  as_gt() # Convierte el objeto gtsummary a un objeto gt tradicional

# Para visualizarla en Colab:
as.tags(tabla_1)

#alternativa markdown
library(knitr)

# La siguiente línea causa un error porque tabla_1 ya es un objeto gt
# tabla_1 %>%
#   as_tibble() %>% # Convierte la tabla a un data frame normal
#   kable()         # La renderiza en texto plano elegante


#alternativa 3

install.packages("modelsummary") # instalan con install.packages("modelsummary") si no lo tienes
library(modelsummary)

# modelsummary detecta automáticamente las variables continuas y categóricas
datasummary_balance(~ grupo, data = datos)

In [ ]:
# ----------------------------------------------------------
# 6. Histograma con línea de media y mediana
# ----------------------------------------------------------

media_glucosa <- mean(datos$glucosa_mg_dl, na.rm = TRUE)
mediana_glucosa <- median(datos$glucosa_mg_dl, na.rm = TRUE)

ggplot(datos, aes(x = glucosa_mg_dl)) +
  geom_histogram(bins = 25, na.rm = TRUE) +
  geom_vline(xintercept = media_glucosa, linetype = "dashed") +
  geom_vline(xintercept = mediana_glucosa, linetype = "dotted") +
  labs(
    title = "Glucosa: histograma con media y mediana",
    subtitle = "Línea discontinua = media; línea punteada = mediana",
    x = "Glucosa (mg/dL)",
    y = "Frecuencia"
  ) +
  theme_minimal()

#**Ejercicios**


**Ejercicio 1:**
 Calcular media ± DE y mediana [Q1, Q3] de colesterol_mg_dl.

**Ejercicio 2:**
Calcular frecuencia de hipertensión por sexo.

**Ejercicio 3:**
Crear una tabla 1 comparando por evento_complicacion.
